In [1]:
from PIL import Image
import requests
from transformers import AutoProcessor, AutoModel
import torch

/home/compu/anaconda3/envs/eagle2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
multiview = True
batch = not multiview

In [3]:
model = AutoModel.from_pretrained("nvidia/Eagle2-2B",trust_remote_code=True, torch_dtype=torch.bfloat16)
processor = AutoProcessor.from_pretrained("nvidia/Eagle2-2B", trust_remote_code=True, use_fast=True)
processor.tokenizer.padding_side = "left"

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.
Some kwargs in processor config are unused and will not have any effect: image_end_token, image_start_token, tokens_per_tile, video_placeholder, image_placeholder, auto_map. 


In [4]:
if multiview:
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": "https://www.ilankelman.org/stopsigns/australia.jpg",
                },
                # {
                #     "type": "image",
                #     "image": "https://www.nvidia.com/content/dam/en-zz/Solutions/about-nvidia/logo-and-brand/01-nvidia-logo-vert-500x200-2c50-d@2x.png",
                # },
                {"type": "text", "text": "Describe these two images."},
            ],
        }
    ]

    text_list = [processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )]
    image_inputs, video_inputs = processor.process_vision_info(messages)
    
if batch:
    messages1 = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": "https://www.ilankelman.org/stopsigns/australia.jpg",
                },
                {"type": "text", "text": "Describe this image."},
            ],
        }
    ]

    messages2 = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": "https://www.nvidia.com/content/dam/en-zz/Solutions/about-nvidia/logo-and-brand/01-nvidia-logo-vert-500x200-2c50-d@2x.png",
                },
                {"type": "text", "text": "Describe this image."},
            ],
        }
    ]

    text_list = [processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ) for messages in [messages1, messages2]]
    image_inputs, video_inputs = processor.process_vision_info([messages1, messages2])

In [5]:
inputs = processor(text = text_list, images=image_inputs, videos=video_inputs, return_tensors="pt", padding=True)
inputs = inputs.to("cuda")
model = model.to("cuda")
generated_ids = model.generate(**inputs, max_new_tokens=1024)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


In [7]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_sizes'])

In [6]:
from PIL import Image
import requests
from io import BytesIO

img_url = "https://www.nvidia.com/content/dam/en-zz/Solutions/about-nvidia/logo-and-brand/01-nvidia-logo-vert-500x200-2c50-d@2x.png"
response = requests.get(img_url)
img = Image.open(BytesIO(response.content))
print("Original image size:", img.size)

Original image size: (1260, 709)


In [7]:
print(inputs.keys())
print(inputs['image_sizes'])
print(inputs['pixel_values'].shape)

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_sizes'])
tensor([[ 876, 1300]], device='cuda:0')
torch.Size([7, 3, 448, 448])


In [6]:
print(text_list)
print(image_inputs)
print(video_inputs)
print(image_inputs[0].size)
print(image_inputs[1].size)

['<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<image-1><image-2>Describe these two images.<|im_end|>\n<|im_start|>assistant\n']
[<PIL.Image.Image image mode=RGB size=1300x876 at 0x7F362552A7A0>, <PIL.Image.Image image mode=RGB size=1260x709 at 0x7F362553CFD0>]
None
(1300, 876)
(1260, 709)


In [6]:
output_text = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['Image-1: The image depicts a street scene with a prominent red stop sign in the foreground, featuring the word "STOP" in white letters. Behind the stop sign, there is a traditional Chinese archway with red and gold colors, adorned with Chinese characters. The archway is flanked by two white stone lion statues. In the background, a black car is driving on the street, and various storefronts with signs in English and Chinese are visible, including one with the word "OPTUS" and another with "KUO." The scene suggests a multicultural urban area, likely a Chinatown district, with a mix of Western and Eastern architectural influences.\n\nImage-2: The image features the logo of NVIDIA, a well-known technology company specializing in graphics processing units (GPUs). The logo consists of a stylized green eye with a white outline, set against a white background. Below the eye, the word "NVIDIA" is written in bold, black capital letters. The overall design is simple and modern, with a clear emp